# 🚦 AI Traffic Vehicle Counter — GPU Runtime (Google Colab)

**Before running:** Go to `Runtime → Change runtime type → T4 GPU` and click Save.

Run all cells top to bottom. The public URL will appear at the end of **Cell 4**.

In [ ]:
# Cell 1 — Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU detected:')
    print(result.stdout.split('\n')[8])  # print GPU name line
else:
    print('❌ No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Cell 2 — Install dependencies
!pip install -q streamlit pyngrok ultralytics opencv-python-headless \
    pandas matplotlib numpy lapx pillow
print('✅ All packages installed')

In [ ]:
# Cell 3 — Clone the project from GitHub
import os
APP_DIR = '/content/ai-traffic-vehicle-counter'

if os.path.exists(APP_DIR):
    !git -C {APP_DIR} pull
else:
    !git clone https://github.com/K-SiddharthKishore/ai-traffic-vehicle-counter {APP_DIR}

print('✅ Project ready at', APP_DIR)

In [ ]:
# Cell 4 — Launch Streamlit + create public URL via ngrok
#
# Get your FREE ngrok token at https://dashboard.ngrok.com/get-started/your-authtoken
# (takes 30 seconds to sign up — no credit card)
# Paste it below:
NGROK_TOKEN = "PASTE_YOUR_TOKEN_HERE"   # <-- replace this

# -----------------------------------------------------------------------
import subprocess, threading, time, os
from pyngrok import ngrok, conf

os.makedirs('/content/outputs', exist_ok=True)

# Configure ngrok
conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()  # close any old tunnels

# Start Streamlit in background thread
def run_streamlit():
    subprocess.run([
        'streamlit', 'run',
        f'{APP_DIR}/app.py',
        '--server.port=8501',
        '--server.headless=true',
        '--server.enableCORS=false',
        '--server.enableXsrfProtection=false',
        '--server.maxUploadSize=500',
    ])

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
time.sleep(6)  # wait for Streamlit to start

# Open ngrok tunnel
tunnel = ngrok.connect(8501)
public_url = tunnel.public_url

print('=' * 60)
print(f'🚀  PUBLIC URL  →  {public_url}')
print('=' * 60)
print('Share this link with your faculty.')
print('Keep this Colab tab open while demonstrating.')
print('GPU (CUDA) will be used automatically for YOLO inference.')

In [ ]:
# Cell 5 (optional) — Print URL again if you scrolled past it
tunnels = ngrok.get_tunnels()
if tunnels:
    print('🔗 App URL:', tunnels[0].public_url)
else:
    print('No tunnel active — re-run Cell 4')